# Direct S3 Access for EarthScope's miniSEED data repository

The SAGE miniSEED data archive is periodically synced to an Amazon S3 bucket.

In this example workflow, a researcher with an EarthScope user account exchanges their EarthScope login credentials for temporary AWS Credentials. These temporary credentials are used to initiate a `boto3` session to access the data in the S3 bucket, through an AccessPoint. You can think of the bucket as a warehouse of data, and the Access Point as a doorway into that warehouse. Anywhere in your script you would use the name of the bucket to access data, you will use the name of the Access Point, and validate your credentials against that Access Point. 

### Pre-requisites:
* The user has registered for an EarthScope user account at https://www.earthscope.org/user/login
* The user's account has been approved/enabled for AWS Credential Brokering
* The user's workflow is configured to run in AWS US-East-2. This script will not work in your local environment. It must be run in your AWS environment or in GeoLab. GeoLab should be used for prototyping only.  


In [1]:
import os
import boto3
import requests
from pathlib import Path

from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
from earthscope_sdk.auth.auth_flow import NoTokensError

## Get Temporary AWS Credentials

In order to talk directly to S3, the user needs AWS Credentials.

https://api.earthscope.org has a new endpoint for brokering temporary (short-lived) AWS credentials with narrowly scoped permission to read from our S3 Access Point serving Restricted and Open Data. [See API documentation](https://api.earthscope.org/beta/docs#get-/user/credentials/aws/-role-)

To use this endpoint (just like every other endpoint in api.earthscope.org), the user needs to pass their oauth2 access token. This can be retrieved 

#### STEP 1: Retrieve Auth0 Access Token with EarthScope SDK Device Flow
Use the cell below to generate a unique authentication code, and prompt the user to navigate to an AWS single-sign-on URL to verify their device. 

In [ ]:
token_path = "./"
device_flow = DeviceCodeFlowSimple(Path(token_path))

# get access token from local path
# try:
#     device_flow.get_access_token_refresh_if_necessary()
# except NoTokensError:
#     # if no token was found locally, do the device code flow
device_flow.do_flow()
print("Got access token")
    
token = device_flow.access_token
print(f"access token expires at {device_flow.expires_at.isoformat()}")

#### STEP 2: Exchange the Auth0 token for temporary AWS Credentials

In [ ]:
# Configure Auth header
headers = {"Authorization": f"Bearer {token}"}

# Retrieve temporary AWS creds
r = requests.get("https://api.earthscope.org/beta/user/credentials/aws/s3-miniseed", headers=headers)
r.raise_for_status()
creds = r.json()

## Create a boto3 Session with the temporary credentials

`boto3` is the AWS SDK for Python. The following cell creates a new boto "Session" with the retrieved temporary credentials.

In [ ]:
session = boto3.Session(
    aws_access_key_id=creds["aws_access_key_id"],
    aws_secret_access_key=creds["aws_secret_access_key"],
    aws_session_token=creds["aws_session_token"],
)

# check my identity
sts = session.client("sts")
print(sts.get_caller_identity())

# create an S3 client
s3_client = session.client("s3")

## Define constants for data location

The data is exposed via an S3 Access Point (S3AP). This S3AP has a unique, generated alias (that we do not control). This alias is used in place of the bucket name in all S3 operations.

EarthScope's miniSEED data lives in the `miniseed/` prefix of our bucket, and thus of our S3AP.

In [ ]:
RESTRICTED_DATA_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"

BUCKET = RESTRICTED_DATA_ACCESS_POINT
PREFIX = "miniseed/"

## List contents of miniseed prefix

Users are allowed to list the `miniseed/` prefix of this access point, including all "subdirectories" within. Even restricted data networks are allowed to be *listed* (but not downloaded).

Listing is permitted from any machine, as long as the user has valid credentials that they obtained from https://api.earthscope.org.

In [ ]:
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
print(nets)

### Consumer function

This function is basically a no-op that simply drains the entire response being read from S3, and counts the number of bytes read

In [ ]:
CHUNK_SIZE = 20_000_000

def read_in_chunks(s3_object: dict):
    """A generator that iterates over an S3 object in chunks."""
    stream = s3_object["Body"]._raw_stream
    ##Insert your data processing here with s3_object
    ct = 0
    while True:
        data = stream.read(CHUNK_SIZE)
        
        if not data:
            break

        ct += len(data)

    return ct

## Trying to access Restricted Data

This is what a user would see if they try to download an object they do not have permission to read.

In [ ]:
try:
    get_resp = s3_client.get_object(
        Bucket=BUCKET,
        Key=f"{PREFIX}BV/2024/090/SOEH.BV.2024.090",
    )
    raise RuntimeError("Should not reach this line")
except Exception as e:
    print("Successfully failed to get restricted data. The following is the error message a user would see:")
    print(e)

## Access Open Data, or Restricted Data the user has access to

If the user tries to download either:
- any Open Data
- any Restricted Data that the user has been granted access to

then the user will *not* see an error message, and instead successfully download the object directly from S3.

This cell has hardcoded a few known Open Data objects for comparing download times across different sized objects.

In [ ]:
# %%timeit

get_resp = s3_client.get_object(
    Bucket=BUCKET,
    Key=f"{PREFIX}UW/2024/300/MBW.UW.2024.300#2",  # 6 MB
    # Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # 85 MB
    # Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # 400 MB
)
sz = read_in_chunks(get_resp)
print(f"Successfully read object from S3 ({sz} bytes)")
